# Project 4 — Deep Learning Systems

## Problem Definition

This project uses deep learning with a Transformer model to detect rental listing risk patterns in text descriptions.

The dataset used is Toronto Kijiji Rental Data from Kaggle. It contains real rental advertisements with listing text and structured rental information.

The goal of this project is to classify listing text into three categories:

- **Safe**
- **Suspicious**
- **High Risk**

These labels reflect potential concerns such as vague wording, misleading claims, restrictive conditions, payment pressure, fraud-like requests, or other renter-risk signals.

## Dataset Source

Toronto Kijiji Rental Data (Kaggle)

https://www.kaggle.com/datasets/umeshkhatiwada/toronto-kijiji-rental-data

In [2]:
import pandas as pd

# Load dataset file
df = pd.read_csv("kijiji_rental_ads_4106.csv")

# Show number of rows and columns
print("Rows and Columns:", df.shape)

# Show all column names
print("\nColumns:")
print(df.columns)

# Show first 5 rows
df.head()

Rows and Columns: (4106, 23)

Columns:
Index(['Title', 'Price($)', 'Address', 'Date Posted', 'Building Type',
       'Bedrooms', 'Bathrooms', 'Utilities', 'Wi-Fi and More',
       'Parking Included', 'Agreement Type', 'Move-In Date', 'Pet Friendly',
       'Size (sqft)', 'Furnished', 'Air Conditioning',
       'Personal Outdoor Space', 'Smoking Permitted', 'Appliances',
       'Amenities', 'Description', 'Visit Counter', 'url'],
      dtype='object')


,Title,Price($),Address,Date Posted,Building Type,Bedrooms,Bathrooms,Utilities,Wi-Fi and More,Parking Included,...,Size (sqft),Furnished,Air Conditioning,Personal Outdoor Space,Smoking Permitted,Appliances,Amenities,Description,Visit Counter,url
0,6020 Bathurst Street - Valencia Towers Apartme...,3209.0,"6020 Bathurst Street, Toronto, ON, M2R 1Z8",2024-02-24 23:30:00,Apartment,2,1,NaN,Not Included,0,...,912,No,No,Not Included,No,Fridge / Freezer,Elevator in Building,Valencia Towers is a student- and family-frien...,NaN,https://www.kijiji.ca/v-apartments-condos/city...
1,RENOVATED BACHELOR SUITE AVAILABLE! Lakeview ...,2000.0,"22 Close Avenue, Toronto, ON, M6K 2V2",2024-03-14 00:09:49,Apartment,Bachelor/Studio,1,"Hydro_No,Heat_No,Water_Yes",Not Included,0,...,445,No,No,Balcony,No,"Laundry (In Building), Fridge / Freezer","Gym, Pool, Storage Space, Elevator in Building","Bachelors, 1 Bath, Recently Renovated Kitchen ...",NaN,https://www.kijiji.ca/v-apartments-condos/city...
2,50 Driftwood - Ruby Heights Apartment for Rent,2819.0,"50 Driftwood, Toronto, ON, M3N 2M6",2024-03-03 00:08:58,Apartment,2,1,NaN,Not Included,0,...,904,No,No,Not Included,No,Fridge / Freezer,"Storage Space, Elevator in Building","Ruby Heights, located in the North York distri...",NaN,https://www.kijiji.ca/v-apartments-condos/city...
3,1 Bed Apartment Rent today,2519.0,"100 Parkway Forest Drive, Toronto, ON, M2J 1L6",2024-03-06 00:35:06,Apartment,1,1,"Hydro_No,Heat_Yes,Water_Yes",Not Included,0,...,665,No,No,Yard,Yes,"Laundry (In Building), Fridge / Freezer","Gym, Pool, 24 Hour Security, Storage Space, El...","For a limited time, you can receive ONE MONTH ...",NaN,https://www.kijiji.ca/v-apartments-condos/city...
4,Brand New 2-bedroom Rental in North York! Yor...,2690.0,"1225 York Road, Toronto, ON, M3A 1Y4",2024-03-04 00:39:41,Apartment,2,1,NaN,Not Included,0,...,Not Available,No,Yes,Balcony,Yes,"Laundry (In Unit), Dishwasher, Fridge / Freezer","Gym, Bicycle Parking, Storage Space, Elevator ...",Realstar's ONE225 York Mills is North Yorks ne...,NaN,https://www.kijiji.ca/v-apartments-condos/city...


## Dataset Inspection Notes

The dataset contains 4,106 rows and 23 columns, which is enough for a deep learning text classification project.

Important columns in this dataset include `Title`, `Price($)`, `Bedrooms`, `Bathrooms`, `Address`, and `Description`.

The most important text column for this project is `Description`, because it contains the main rental listing language that can be used for Transformer-based NLP.

Some columns may need cleaning because they contain missing values, mixed formats, or text that is not useful for the classification task.

In [4]:
# Check data types
print("Data types:")
print(df.dtypes)

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

# Check the main text column
print("\nExample descriptions:")
for i, text in enumerate(df["Description"].dropna().head(3), 1):
    print(f"\nExample {i}:")
    print(text[:500])  # show first 500 characters only

Data types:
Title                      object
Price($)                  float64
Address                    object
Date Posted                object
Building Type              object
Bedrooms                   object
Bathrooms                  object
Utilities                  object
Wi-Fi and More             object
Parking Included           object
Agreement Type             object
Move-In Date               object
Pet Friendly               object
Size (sqft)                object
Furnished                  object
Air Conditioning           object
Personal Outdoor Space     object
Smoking Permitted          object
Appliances                 object
Amenities                  object
Description                object
Visit Counter              object
url                        object
dtype: object

Missing values:
Title                        7
Price($)                   256
Address                     12
Date Posted                608
Building Type             1025
Bedrooms            

## Dataset Inspection Notes

- Most columns in the dataset are stored in text format, while the rental price column is numeric.

- Several columns contain missing values, especially optional property features such as amenities, utilities, and move-in date.

- The most important column for this project is `Description`, because it contains the main rental listing text that will be used for Transformer-based classification.

- The example descriptions show real rental language, promotional wording, contact requests, and housing details. This makes the dataset suitable for natural language processing.

- Before modeling, rows with missing descriptions will be removed and text data will be cleaned.

## Data Cleaning and Text Preparation

The dataset was cleaned before deep learning modeling.

The main text column used for this project is `Description`.

Rows with missing descriptions were removed, duplicate text entries were removed, and extra spaces were cleaned from the text.

In [5]:
# Make a clean copy of the dataset
df_clean = df.copy()

# Keep only rows that have description text
df_clean = df_clean.dropna(subset=["Description"])

# Remove duplicate descriptions
df_clean = df_clean.drop_duplicates(subset=["Description"])

# Convert description to string
df_clean["Description"] = df_clean["Description"].astype(str)

# Remove extra spaces
df_clean["Description"] = df_clean["Description"].str.replace(r"\s+", " ", regex=True).str.strip()

# Show updated shape
print("Rows and Columns after cleaning:", df_clean.shape)

# Show first few cleaned rows
df_clean[["Title", "Description"]].head()

Rows and Columns after cleaning: (2796, 23)


,Title,Description
0,6020 Bathurst Street - Valencia Towers Apartme...,Valencia Towers is a student- and family-frien...
1,RENOVATED BACHELOR SUITE AVAILABLE! Lakeview ...,"Bachelors, 1 Bath, Recently Renovated Kitchen ..."
2,50 Driftwood - Ruby Heights Apartment for Rent,"Ruby Heights, located in the North York distri..."
3,1 Bed Apartment Rent today,"For a limited time, you can receive ONE MONTH ..."
4,Brand New 2-bedroom Rental in North York! Yor...,Realstar's ONE225 York Mills is North Yorks ne...


## Public Guidance Used for Annotation Framework

To support research-grade labeling, publicly available Canadian rental safety and fraud guidance was reviewed from three sources:

- Competition Bureau Canada
- RCMP Canada
- University of Toronto fraud prevention guidance

The page text was collected using `trafilatura` and reviewed to identify common renter-risk signals, such as deposit requests before viewing, money transfer requests, personal information requests, misleading urgency language, and lack of formal lease processes.

These sources were used as reference material to guide manual annotation of rental listing descriptions rather than to automatically generate labels.

In [13]:
# Install required libraries
!pip install trafilatura

In [14]:
# If needed, install once:
# !pip install trafilatura

import trafilatura

# Public Canadian guidance sources
urls = [
    "https://www.canada.ca/en/competition-bureau/news/2018/08/rental-scam-no-room-for-error.html",
    "https://rcmp.ca/en/bc/safety-tips/frauds-and-scams/rental-scams",
    "https://www.communitysafety.utoronto.ca/fraud-prevention/fraud-prevention-tips/fraud-prevention-tip-landlord-or-rental-scams/"
]

# Collect text
all_text = ""

for url in urls:
    downloaded = trafilatura.fetch_url(url)
    extracted_text = trafilatura.extract(downloaded)

    if extracted_text:
        all_text += extracted_text + "\n\n"

# Show preview of reference material
print(all_text[:3000])

Rental scam: no room for error
News release
August 16, 2018 – OTTAWA, ON – Competition Bureau
It’s peak moving season. Students are leaving the nest; parents are helping them find the right place. Beware: if a rental listing looks too good to be true, it probably is. School might not have started yet, but do your homework and learn to recognize rental scams.
In a typical rental scam, fraudsters will entice you with a very attractive listing: sought after area, great amenities and low price. Ads will be posted on popular sites like Kijiji or Facebook. Scammers may use photos from an old listing, from a house that’s up for sale, or from short-term rental sites like Airbnb, to make it look authentic. They pose as the landlord and may claim to be abroad and unable to meet in person to show you inside the place.
After a few emails or text messages, they will start asking for money. First, they’ll try to get a security deposit, then, they’ll ask for the first month’s rent, and then another m

## Final Annotation Framework

This project uses a broader rental listing risk detection framework rather than scam-only detection.

Each listing is manually labeled into one of three classes:

- **Safe**: Normal rental listing language with no clear concerning signals.
- **Suspicious**: Low-trust, vague, restrictive, misleading, or unusual wording that may require caution.
- **High Risk**: Strong concerning signals such as payment pressure, fraud-like requests, exploitative restrictions, or serious fairness concerns.

This broader framework better reflects real rental risks faced by renters in practice.

In [15]:
# Keep only rows with usable descriptions
annot_df = df_clean[["Title", "Description"]].copy()

# Reset index
annot_df = annot_df.reset_index(drop=True)

# Take first 30 rows for manual annotation starter set
sample_df = annot_df.head(30).copy()

# Add empty label column
sample_df["Risk_Label"] = ""

# Show rows for annotation
sample_df[["Title", "Description", "Risk_Label"]]

,Title,Description,Risk_Label
0,6020 Bathurst Street - Valencia Towers Apartme...,Valencia Towers is a student- and family-frien...,
1,RENOVATED BACHELOR SUITE AVAILABLE! Lakeview ...,"Bachelors, 1 Bath, Recently Renovated Kitchen ...",
2,50 Driftwood - Ruby Heights Apartment for Rent,"Ruby Heights, located in the North York distri...",
3,1 Bed Apartment Rent today,"For a limited time, you can receive ONE MONTH ...",
4,Brand New 2-bedroom Rental in North York! Yor...,Realstar's ONE225 York Mills is North Yorks ne...,
5,Furnished room available (Eglinton West Statio...,Clean 3 bedroom furnished apartment. Room avai...,
6,Collegiate Court - 1 Bedroom Apartment for Rent,"A clean, quiet and well-maintained rental buil...",
7,Chatsworth Apartments - 1 Bdrm available at 29...,1 Month Free Rent,
8,Basement for rent in Scarborough,2 bedrooms 1 washroom full renovate basement a...,
9,One Bedroom condo available,Discover the epitome of modern living in our o...,


## Manual Annotation Process

The first sample of listings was manually reviewed using Canadian public rental guidance and broader renter-risk criteria.

Labels used:

- **Safe** = normal listing language with no clear concern  
- **Suspicious** = vague wording, restrictive conditions, misleading claims, unusual requests, or low-trust signals  
- **High Risk** = strong concerning signals such as fraud-like requests, payment pressure, exploitative restrictions, or serious renter-risk indicators

In [16]:
# Manually assign first 10 labels
sample_df.loc[0, "Risk_Label"] = "Safe"
sample_df.loc[1, "Risk_Label"] = "Safe"
sample_df.loc[2, "Risk_Label"] = "Safe"
sample_df.loc[3, "Risk_Label"] = "Suspicious"
sample_df.loc[4, "Risk_Label"] = "Safe"
sample_df.loc[5, "Risk_Label"] = "Safe"
sample_df.loc[6, "Risk_Label"] = "Safe"
sample_df.loc[7, "Risk_Label"] = "Suspicious"
sample_df.loc[8, "Risk_Label"] = "Safe"
sample_df.loc[9, "Risk_Label"] = "Safe"

# Show first 10 labeled rows
sample_df.head(10)

,Title,Description,Risk_Label
0,6020 Bathurst Street - Valencia Towers Apartme...,Valencia Towers is a student- and family-frien...,Safe
1,RENOVATED BACHELOR SUITE AVAILABLE! Lakeview ...,"Bachelors, 1 Bath, Recently Renovated Kitchen ...",Safe
2,50 Driftwood - Ruby Heights Apartment for Rent,"Ruby Heights, located in the North York distri...",Safe
3,1 Bed Apartment Rent today,"For a limited time, you can receive ONE MONTH ...",Suspicious
4,Brand New 2-bedroom Rental in North York! Yor...,Realstar's ONE225 York Mills is North Yorks ne...,Safe
5,Furnished room available (Eglinton West Statio...,Clean 3 bedroom furnished apartment. Room avai...,Safe
6,Collegiate Court - 1 Bedroom Apartment for Rent,"A clean, quiet and well-maintained rental buil...",Safe
7,Chatsworth Apartments - 1 Bdrm available at 29...,1 Month Free Rent,Suspicious
8,Basement for rent in Scarborough,2 bedrooms 1 washroom full renovate basement a...,Safe
9,One Bedroom condo available,Discover the epitome of modern living in our o...,Safe


In [17]:
# Manually assign rows 10 to 19
sample_df.loc[10, "Risk_Label"] = "Safe"
sample_df.loc[11, "Risk_Label"] = "Suspicious"
sample_df.loc[12, "Risk_Label"] = "Suspicious"
sample_df.loc[13, "Risk_Label"] = "Safe"
sample_df.loc[14, "Risk_Label"] = "Safe"
sample_df.loc[15, "Risk_Label"] = "Safe"
sample_df.loc[16, "Risk_Label"] = "Suspicious"
sample_df.loc[17, "Risk_Label"] = "Suspicious"
sample_df.loc[18, "Risk_Label"] = "Safe"
sample_df.loc[19, "Risk_Label"] = "Safe"

# Show rows 10 to 19
sample_df.loc[10:19]

,Title,Description,Risk_Label
10,Large renovated 1 Bedroom with balcony at Bath...,Large renovated 1 bedroom with balcony and har...,Safe
11,Queen and woodbine beaches,2 bed 2 nd floor apartment for rent Available ...,Suspicious
12,3 Bedroom Apartment for Rent - 3434 Eglinton A...,Utilities Included,Suspicious
13,"2 Bed, 1 Bath Basement Apartment for Rent in W...",Thanks for reading. Up for rent is a 2 bedroom...,Safe
14,1 Bed 1 Bath - Apartment,"Incredible loft-style living in a remarkable, ...",Safe
15,One bedroom apartment for rent,Pickering one bedroom apartment for rent. Sepa...,Safe
16,105 West Lodge - 1 Bedroom Apartment in the He...,Building Overivew,Suspicious
17,"HUGE 50' x 50' yard. ALL main house. Prime, sa...","**FREE WIFI, PARKING and ALL CURTAINS included...",Suspicious
18,Newly Renovated 1 Bedroom Basement for rent,Brand new legal basement apartment with separa...,Safe
19,Subletting a Cozy and Stylish Dorm Room for 2 ...,Dorm room available for 2 months from June 1st...,Safe


In [18]:
# Manually assign rows 20 to 29
sample_df.loc[20, "Risk_Label"] = "Suspicious"
sample_df.loc[21, "Risk_Label"] = "Safe"
sample_df.loc[22, "Risk_Label"] = "Safe"
sample_df.loc[23, "Risk_Label"] = "Safe"
sample_df.loc[24, "Risk_Label"] = "Suspicious"
sample_df.loc[25, "Risk_Label"] = "Safe"
sample_df.loc[26, "Risk_Label"] = "Safe"
sample_df.loc[27, "Risk_Label"] = "Safe"
sample_df.loc[28, "Risk_Label"] = "High Risk"
sample_df.loc[29, "Risk_Label"] = "Safe"

# Show rows 20 to 29
sample_df.loc[20:29]

,Title,Description,Risk_Label
20,"New Two-Bedroom Townhome Rentals- Move In Now,...","Move In Now, Don't Pay Until May 1stOpen House...",Suspicious
21,Furnished 2 bedrooms Basement Apartment - Rich...,Looking for a comfortable and well-equipped li...,Safe
22,Room for rent,Bechelorstudio apartment Markham/ Eglinton (Sc...,Safe
23,Spacious Apartment in South Parkdale-All utili...,Best All Inclusive Lease In South Parkdale. Ne...,Safe
24,Golden Mile 1 Bedroom Apartment for Rent - 32 ...,Current Promotions,Suspicious
25,"Open Concept Loft, 18' ceiling, Live/Work Studio","Perfect for Yoga Studio, Creative Agency, Phot...",Safe
26,Large 1 Bed Room Basement Apartment Prime Loca...,"Prime location Queen St. East, downtown Toront...",Safe
27,2 bedroom apartment for rent sublease,Near weston and finchUtilities and car parking...,Safe
28,FINCH&BATHURST-MALE ONLY-MONTH TO MONTH RENTAL...,"Welcome,",High Risk
29,"2 bdrm +den - Fifty on the Park,","2 bdrm +den , 1 bath east view3 black applianc...",Safe
